In [1]:
import pandas as pd
import sys
sys.path.append('..')
from src.config import CLEAN_DATA_PATH

df = pd.read_csv(CLEAN_DATA_PATH, parse_dates=['InvoiceDate'])
trading_days = df['InvoiceDate'].dt.normalize().drop_duplicates().sort_values().reset_index(drop=True)
gaps = trading_days.diff().dt.days

closures = pd.DataFrame({
    'last_trading_day': trading_days.shift(1)[gaps > 3],
    'next_trading_day': trading_days[gaps > 3],
    'gap_days': gaps[gaps > 3]
})
print(closures)

    last_trading_day next_trading_day  gap_days
21        2009-12-23       2010-01-04      12.0
97        2010-04-01       2010-04-06       5.0
319       2010-12-23       2011-01-04      12.0
412       2011-04-21       2011-04-26       5.0


In [2]:
import pandas as pd
import sys
sys.path.append('..')
from src.config import NON_PRODUCT_CODES

raw = pd.concat(
    pd.read_excel('../data/online_retail_II.xlsx', sheet_name=None).values(),
    ignore_index=True
)
raw['Invoice'] = raw['Invoice'].astype(str)
raw['Is_Cancel'] = raw['Invoice'].str.startswith('C')

cols = ['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'InvoiceDate']

# 1. Customer 12346: is the 74,215-unit order followed by a cancellation?
print("Customer 12346, 2011 transactions:")
print(raw[(raw['Customer ID'] == 12346) & (raw['InvoiceDate'] >= '2011-01-01')][cols])

# 2. The 80,995-unit order: who placed it, and was it cancelled?
big = raw[raw['Quantity'].abs() == 80995][cols + ['Customer ID']]
print("\nAll lines with |Quantity| = 80,995:")
print(big)

# 3. Overall scale of cancellations (product lines with a Customer ID only)
product = ~raw['StockCode'].astype(str).isin(NON_PRODUCT_CODES) & raw['Customer ID'].notna()
cancels = raw[product & raw['Is_Cancel']]
sales = raw[product & ~raw['Is_Cancel'] & (raw['Quantity'] > 0) & (raw['Price'] > 0)]

cancel_value = (-cancels['Quantity'] * cancels['Price']).sum()
sales_value = (sales['Quantity'] * sales['Price']).sum()
print(f"\nCancellation lines: {len(cancels):,}")
print(f"Cancelled value: £{cancel_value:,.0f}")
print(f"Gross sales value: £{sales_value:,.0f}")
print(f"Cancelled as % of gross sales: {cancel_value / sales_value * 100:.1f}%")

Customer 12346, 2011 transactions:
        Invoice StockCode                     Description  Quantity  Price  \
587080   541431     23166  MEDIUM CERAMIC TOP STORAGE JAR     74215   1.04   
587085  C541433     23166  MEDIUM CERAMIC TOP STORAGE JAR    -74215   1.04   

               InvoiceDate  
587080 2011-01-18 10:01:00  
587085 2011-01-18 10:17:00  

All lines with |Quantity| = 80,995:
         Invoice StockCode                  Description  Quantity  Price  \
1065882   581483     23843  PAPER CRAFT , LITTLE BIRDIE     80995   2.08   
1065883  C581484     23843  PAPER CRAFT , LITTLE BIRDIE    -80995   2.08   

                InvoiceDate  Customer ID  
1065882 2011-12-09 09:15:00      16446.0  
1065883 2011-12-09 09:27:00      16446.0  

Cancellation lines: 17,983
Cancelled value: £729,146
Gross sales value: £17,438,960
Cancelled as % of gross sales: 4.2%


In [3]:
df = raw.drop_duplicates()
product = ~df['StockCode'].astype(str).isin(NON_PRODUCT_CODES) & df['Customer ID'].notna()

cancels = df[product & df['Is_Cancel']].copy()
sales = df[product & ~df['Is_Cancel'] & (df['Quantity'] > 0) & (df['Price'] > 0)].copy()

cancels['Abs_Qty'] = -cancels['Quantity']
cancels['Value'] = cancels['Abs_Qty'] * cancels['Price']
cancels = cancels.reset_index().rename(columns={'index': 'cancel_idx'})
sales = sales.reset_index().rename(columns={'index': 'sale_idx'})

keys = ['Customer ID', 'StockCode']

# Case 1: exact pairs (same customer, product, quantity, price; purchase at or before cancellation)
exact = cancels.merge(
    sales[keys + ['Quantity', 'Price', 'InvoiceDate', 'sale_idx']],
    left_on=keys + ['Abs_Qty', 'Price'],
    right_on=keys + ['Quantity', 'Price'],
    suffixes=('', '_sale')
)
exact = exact[exact['InvoiceDate_sale'] <= exact['InvoiceDate']]
exact_ids = set(exact['cancel_idx'])

# Case 2 vs 3: among the rest, does any earlier purchase of that product by that customer exist?
rest = cancels[~cancels['cancel_idx'].isin(exact_ids)]
prior = rest.merge(sales[keys + ['InvoiceDate']], on=keys, suffixes=('', '_sale'))
prior = prior[prior['InvoiceDate_sale'] <= prior['InvoiceDate']]
partial_ids = set(prior['cancel_idx'])

groups = {
    'Case 1: exact pair': cancels[cancels['cancel_idx'].isin(exact_ids)],
    'Case 2: partial / non-exact': rest[rest['cancel_idx'].isin(partial_ids)],
    'Case 3: unmatched': rest[~rest['cancel_idx'].isin(partial_ids)],
}

total = cancels['Value'].sum()
print(f"Total cancelled value (deduplicated): £{total:,.0f}\n")
for name, part in groups.items():
    v = part['Value'].sum()
    print(f"{name:30s} lines: {len(part):6,}   value: £{v:>10,.0f}   ({v / total * 100:5.1f}%)")

print("\nTop 10 cancellations by value:")
print(cancels.nlargest(10, 'Value')[['Invoice', 'Customer ID', 'Description', 'Abs_Qty', 'Price', 'Value', 'InvoiceDate']])

# Where does the fake £168k currently sit in the modeling data?
from src.config import MODELING_DATA_PATH
mt = pd.read_csv('..' + '/' + MODELING_DATA_PATH if not MODELING_DATA_PATH.startswith('..') else MODELING_DATA_PATH)
if 'Customer ID' in mt.columns:
    mt = mt.set_index('Customer ID')
print("\nTop 5 CLV_Target values (6-month table):")
print(mt['CLV_Target'].nlargest(5))

Total cancelled value (deduplicated): £719,446

Case 1: exact pair             lines:  6,107   value: £   548,070   ( 76.2%)
Case 2: partial / non-exact    lines:  9,953   value: £   132,312   ( 18.4%)
Case 3: unmatched              lines:  1,576   value: £    39,064   (  5.4%)

Top 10 cancellations by value:
       Invoice  Customer ID                         Description  Abs_Qty  \
17630  C581484      16446.0         PAPER CRAFT , LITTLE BIRDIE    80995   
10073  C541433      12346.0      MEDIUM CERAMIC TOP STORAGE JAR    74215   
11811  C550456      15749.0  FAIRY CAKE FLANNEL ASSORTED COLOUR     3114   
11809  C550456      15749.0  WHITE HANGING HEART T-LIGHT HOLDER     1930   
11807  C550456      15749.0                  DOORMAT FAIRY CAKE      670   
14807  C567527      17450.0               PANTRY CHOPPING BOARD      756   
11810  C550456      15749.0         GIN + TONIC DIET METAL SIGN     2000   
1818   C498151      13902.0     SMALL FAIRY CAKE FRIDGE MAGNETS     2504   
11808